# Notebook 01 — SQL Server → MinIO Landing Zone (CSV)

Extrai **todas as 11 tabelas** do banco `DespesaPublicaDB` via **PySpark JDBC** e grava cada uma como CSV no bucket `landing-zone` do MinIO.

**Sem ODBC.** A leitura do SQL Server é feita 100% via JDBC pelo Spark.

```
SQL Server (DespesaPublicaDB)  ──►  PySpark JDBC  ──►  MinIO / landing-zone / tabela / tabela.csv
```

In [ ]:
import os
import boto3
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv()

SERVER   = os.getenv('SQL_SERVER',   'localhost')
PORT     = os.getenv('SQL_PORT',     '1433')
DATABASE = os.getenv('SQL_DATABASE', 'DespesaPublicaDB')
USER     = os.getenv('SQL_USER',     'sa')
PASSWORD = os.getenv('SQL_PASSWORD', 'SqlServer@2024!')

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT',   'http://localhost:9020')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
BUCKET_LANDING   = os.getenv('MINIO_BUCKET_LANDING', 'landing-zone')

JDBC_URL = (
    f'jdbc:sqlserver://{SERVER}:{PORT};'
    f'databaseName={DATABASE};encrypt=true;trustServerCertificate=true'
)
JDBC_PROPS = {
    'user': USER, 'password': PASSWORD,
    'driver': 'com.microsoft.sqlserver.jdbc.SQLServerDriver'
}

print(f'SQL Server : {SERVER}:{PORT} / {DATABASE}')
print(f'MinIO      : {MINIO_ENDPOINT} -> {BUCKET_LANDING}')


SQL Server : localhost:1433 / DespesaPublicaDB
MinIO      : http://localhost:9020 -> landing-zone


## 1. SparkSession — JDBC SQL Server + Hadoop S3A (MinIO)

In [ ]:
spark = (
    SparkSession.builder
    .appName('DespesaPublica-01-SQLServer-to-MinIO')
    .config(
        'spark.jars.packages',
        'com.microsoft.sqlserver:mssql-jdbc:12.4.2.jre11,'
        'org.apache.hadoop:hadoop-aws:3.3.4,'
        'com.amazonaws:aws-java-sdk-bundle:1.12.367'
    )
    .config('spark.hadoop.fs.s3a.endpoint',          MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key',        MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key',        MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} iniciado')


26/05/05 13:30:14 WARN Utils: Your hostname, DESKTOP-6KNCQNJ resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/05 13:30:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/mendax/spark-delta-minio-despesa/spark-delta-minio-despesa/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/mendax/.ivy2/cache
The jars for the packages stored in: /home/mendax/.ivy2/jars
com.microsoft.sqlserver#mssql-jdbc added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3b4a5c03-a8eb-4c43-9c21-f995d5bcb359;1.0
	confs: [default]
	found com.microsoft.sqlserver#mssql-jdbc;12.4.2.jre11 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.367 in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (178ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.367/aws-java-sdk-bundle-1.12.367.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.

Spark 3.5.3 iniciado


## 2. Criar bucket `landing-zone` no MinIO

In [ ]:
s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY
)

existing = [b['Name'] for b in s3.list_buckets()['Buckets']]
if BUCKET_LANDING not in existing:
    s3.create_bucket(Bucket=BUCKET_LANDING)
    print(f"Bucket '{BUCKET_LANDING}' criado!")
else:
    print(f"Bucket '{BUCKET_LANDING}' ja existe.")


Bucket 'landing-zone' criado!


## 3. Extração: SQL Server → MinIO (CSV)

Cada tabela é lida via JDBC e gravada como CSV em `s3a://landing-zone/<tabela>/<tabela>.csv`.

In [ ]:
TABELAS = [
    'orgao', 'unidade', 'programa', 'acao', 'fonte_recurso',
    'credor', 'empenho', 'item_empenho', 'liquidacao', 'pagamento', 'historico_preco'
]

print('=== EXTRACAO: SQL Server -> MinIO landing-zone ===\n')

for tabela in TABELAS:
    # Ler do SQL Server via JDBC
    df = spark.read.jdbc(url=JDBC_URL, table=tabela, properties=JDBC_PROPS)

    # Gravar como CSV no MinIO (1 arquivo por tabela, sem particoes extras)
    csv_path = f's3a://{BUCKET_LANDING}/{tabela}'
    (
        df.coalesce(1)           # garante 1 único arquivo CSV por tabela
        .write
        .mode('overwrite')
        .option('header', 'true')
        .csv(csv_path)
    )
    print(f'  {tabela:20s} {df.count():4d} registros -> s3a://{BUCKET_LANDING}/{tabela}/')

print('\nExtracao concluida!')


=== EXTRACAO: SQL Server -> MinIO landing-zone ===



26/05/05 13:30:57 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/05 13:31:00 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/05 13:31:02 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
                                                                                

  orgao                   5 registros -> s3a://landing-zone/orgao/


26/05/05 13:31:04 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/05 13:31:04 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
                                                                                

  unidade                11 registros -> s3a://landing-zone/unidade/


...

  liquidacao             40 registros -> s3a://landing-zone/liquidacao/


26/05/05 13:31:12 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/05 13:31:12 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


  pagamento              35 registros -> s3a://landing-zone/pagamento/


26/05/05 13:31:13 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/05 13:31:13 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


  historico_preco        15 registros -> s3a://landing-zone/historico_preco/

Extracao concluida!


## 4. Validação — listar objetos no MinIO

In [ ]:
print(f"=== OBJETOS NO BUCKET '{BUCKET_LANDING}' ===\n")

resp = s3.list_objects_v2(Bucket=BUCKET_LANDING)
total_bytes = 0
for obj in resp.get('Contents', []):
    total_bytes += obj['Size']
    print(f"  {obj['Key']:55s}  {obj['Size']/1024:6.1f} KB")

print(f"\nTotal: {len(resp.get('Contents',[]))} objetos | {total_bytes/1024:.1f} KB")


=== OBJETOS NO BUCKET 'landing-zone' ===

  acao/_SUCCESS                                               0.0 KB
  acao/part-00000-9eb46532-9d39-4e03-ad26-d9338a68955e-c000.csv     0.6 KB
  credor/_SUCCESS                                             0.0 KB
  credor/part-00000-f32d6b8b-7600-43ca-ac64-a2b831b4fb51-c000.csv     0.9 KB
  empenho/_SUCCESS                                            0.0 KB
  empenho/part-00000-c9dc7245-826d-4aab-90e1-032688106ceb-c000.csv     6.3 KB
  fonte_recurso/_SUCCESS                                      0.0 KB
  fonte_recurso/part-00000-c95e5165-50ff-47e4-980f-9c9e656b9ce0-c000.csv     0.2 KB
  historico_preco/_SUCCESS                                    0.0 KB
  historico_preco/part-00000-fda46b70-8186-4045-9a46-1c5a0d826235-c000.csv     0.8 KB
  item_empenho/_SUCCESS                                       0.0 KB
  item_empenho/part-00000-e87219e7-5b28-4e19-ab92-e763ce75e1fc-c000.csv     6.0 KB
  liquidacao/_SUCCESS                                        

## 5. Preview com PySpark — ler de volta do MinIO

In [ ]:
for tabela in ['empenho', 'orgao', 'credor']:
    print(f'\n=== {tabela.upper()} ===')
    df_back = (
        spark.read
        .option('header', 'true')
        .option('inferSchema', 'true')
        .csv(f's3a://{BUCKET_LANDING}/{tabela}')
    )
    df_back.printSchema()
    df_back.show(5, truncate=40)

spark.stop()
print('Notebook 01 concluido!')



=== EMPENHO ===
root
 |-- id_empenho: integer (nullable = true)
 |-- numero_empenho: string (nullable = true)
 |-- id_unidade: integer (nullable = true)
 |-- id_acao: integer (nullable = true)
 |-- id_fonte: integer (nullable = true)
 |-- id_credor: integer (nullable = true)
 |-- data_empenho: date (nullable = true)
 |-- valor_empenho: double (nullable = true)
 |-- tipo_empenho: string (nullable = true)
 |-- modalidade_licitacao: string (nullable = true)
 |-- situacao: string (nullable = true)
 |-- descricao: string (nullable = true)

+----------+--------------+----------+-------+--------+---------+------------+-------------+------------+--------------------+---------+----------------------------------------+
|id_empenho|numero_empenho|id_unidade|id_acao|id_fonte|id_credor|data_empenho|valor_empenho|tipo_empenho|modalidade_licitacao| situacao|                               descricao|
+----------+--------------+----------+-------+--------+---------+------------+-------------+----------